# LeetCode 1891 — Cutting Ribbons bằng Python

Link đề: https://leetcode.com/problems/cutting-ribbons/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search

> **Bài Premium.** Đề bài trong notebook được viết lại theo hiểu biết về bài (nguồn: bản mô tả của doocs/leetcode). Nếu có tài khoản Premium, hãy đối chiếu lại đề gốc trên LeetCode.

Skeleton LeetCode:

```python
class Solution:
    def maxLength(self, ribbons: List[int], k: int) -> int:
        ...
```

Lộ trình:
1. **Thử mọi độ dài `x` từ lớn xuống nhỏ**, đếm số đoạn `Σ ⌊r / x⌋` (`O(max · n)`).
2. **Tìm kiếm nhị phân trên đáp án `x`** — hàm đếm **không tăng** theo `x` (`O(n log max)`).
3. **Thu hẹp cận trên** `x ≤ min(max(ribbons), ⌊Σ ribbons / k⌋)` để giảm số vòng nhị phân và trả `0` sớm.
4. **Đếm theo tần suất + tổng điều hòa**: với bảng đếm độ dài, `count(x)` tính trong `O(V / x)` ⇒ quét mọi `x` từ lớn xuống chỉ `O(V log V)` — không cần nhị phân.

## 1. Mục tiêu học

1. Nhận dạng bài toán **"tìm giá trị lớn nhất thỏa điều kiện đơn điệu"** và áp **tìm kiếm nhị phân trên đáp án**.
2. Viết hàm kiểm tra `count(x) = Σ ⌊rᵢ / x⌋ ≥ k` và chứng minh tính **đơn điệu**.
3. Cài nhị phân "tìm `x` lớn nhất thỏa" đúng biên (tránh vòng lặp vô hạn với `mid = (lo + hi + 1) // 2`).
4. Biết kỹ thuật **tổng điều hòa** `Σ V/x = O(V log V)` khi đếm trên bội số.
5. Kiểm chứng bằng quét tuyến tính trên input nhỏ, xử lý trường hợp **không thể** (trả `0`).

## 2. Đề bài & ví dụ

Cho mảng `ribbons` (`ribbons[i]` là độ dài dải ruy-băng thứ `i`) và số nguyên `k`. Bạn có thể cắt mỗi dải thành bao nhiêu đoạn tùy ý với độ dài **nguyên dương**, hoặc không cắt.

Ví dụ dải dài 4 có thể: giữ nguyên `4`; cắt `3 + 1`; `2 + 2`; `2 + 1 + 1`; hoặc `1 + 1 + 1 + 1`.

Tìm độ dài **lớn nhất** `x` sao cho cắt được **ít nhất `k`** đoạn, mỗi đoạn dài **đúng `x`** (phần thừa bỏ đi). Nếu không thể cắt `k` đoạn cùng độ dài, trả về `0`.

**Ví dụ 1:** `ribbons = [9,7,5], k = 3` → `5` (`9 → 5 + 4`, `7 → 5 + 2`, giữ `5`).

**Ví dụ 2:** `ribbons = [7,5,9], k = 4` → `4` (`7 → 4+3`, `5 → 4+1`, `9 → 4+4+1`).

**Ví dụ 3:** `ribbons = [5,7,9], k = 22` → `0` (tổng chỉ `21 < 22`).

**Ràng buộc:** `1 ≤ ribbons.length ≤ 10⁵`, `1 ≤ ribbons[i] ≤ 10⁵`, `1 ≤ k ≤ 10⁹`.

In [1]:
import random
import time
from typing import List

examples = [
    (([9, 7, 5], 3), 5),
    (([7, 5, 9], 4), 4),
    (([5, 7, 9], 22), 0),
]
for (rib, k), want in examples:
    print(f"ribbons={rib}, k={k} -> kỳ vọng {want}")

ribbons=[9, 7, 5], k=3 -> kỳ vọng 5
ribbons=[7, 5, 9], k=4 -> kỳ vọng 4
ribbons=[5, 7, 9], k=22 -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1 Số đoạn độ dài `x` cắt được từ một dải

Từ dải dài `r` cắt được `⌊r / x⌋` đoạn dài `x` (phần dư `r mod x` bỏ). Phép chia nguyên trong Python: `r // x`.

In [2]:
r = 9
print([(x, r // x) for x in range(1, 10)])

[(1, 9), (2, 4), (3, 3), (4, 2), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1)]


### 3.2 Tổng số đoạn và tính đơn điệu

`count(x) = Σ ⌊rᵢ / x⌋`. Khi `x` tăng, mỗi `⌊rᵢ / x⌋` **không tăng** ⇒ `count(x)` không tăng. Do đó tập `{x : count(x) ≥ k}` có dạng `{1, 2, …, X*}` (hoặc rỗng).

In [3]:
def count(ribbons, x):
    return sum(r // x for r in ribbons)

rib = [9, 7, 5]
print([(x, count(rib, x)) for x in range(1, 10)])

[(1, 21), (2, 9), (3, 6), (4, 4), (5, 3), (6, 2), (7, 2), (8, 1), (9, 1)]


### 3.3 Khuôn nhị phân "tìm `x` LỚN NHẤT thỏa `ok(x)`"

Với `ok` đúng trên `[lo, X*]` và sai trên `(X*, hi]`:

```text
while lo < hi:
    mid = (lo + hi + 1) // 2     # làm tròn LÊN để tránh lặp vô hạn
    if ok(mid): lo = mid
    else: hi = mid - 1
```

Nếu dùng `mid = (lo + hi) // 2` với `lo = mid`, khi `hi = lo + 1` và `ok(lo)` đúng thì `mid = lo` — vòng lặp không tiến.

In [4]:
def last_true(lo, hi, ok):
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if ok(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo

print(last_true(0, 100, lambda x: x * x <= 50), "(căn nguyên của 50)")

7 (căn nguyên của 50)


### 3.4 Tổng điều hòa

`Σ_{x=1}^{V} V/x ≈ V · ln V`. Nếu tính `count(x)` trong `O(V / x)` cho mọi `x`, tổng chi phí `O(V log V)` — với `V = 10⁵` khoảng `1.2·10⁶`.

In [5]:
import math
V = 10**5
print("Σ V/x =", sum(V // x for x in range(1, V + 1)), "≈ V ln V =", int(V * math.log(V)))

Σ V/x = 1166750 ≈ V ln V = 1151292


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 2 `ribbons = [7, 5, 9]`, `k = 4`:

| x | 7 // x | 5 // x | 9 // x | count | ≥ 4? |
|---|---|---|---|---|---|
| 5 | 1 | 1 | 1 | 3 | không |
| 4 | 1 | 1 | 2 | **4** | có |
| 3 | 2 | 1 | 3 | 6 | có |

`x = 4` là giá trị lớn nhất còn đủ ⇒ đáp án `4`. Từ `x = 4` trở xuống luôn đủ (đơn điệu), từ `x = 5` trở lên luôn thiếu.

**Cận trên tự nhiên:** `x ≤ max(ribbons)` (đoạn không dài hơn dải dài nhất) và `x ≤ ⌊Σ ribbons / k⌋` (tổng chiều dài phải đủ cho `k` đoạn). Nếu cận này `= 0` ⇒ không thể ⇒ `0`.

## 5. Cách 1 — Brute Force / baseline: thử mọi `x` từ lớn xuống

In [6]:
def max_length_linear(ribbons, k):
    for x in range(max(ribbons), 0, -1):
        if sum(r // x for r in ribbons) >= k:
            return x
    return 0

for (rib, k), want in examples:
    print(max_length_linear(rib, k), "kỳ vọng", want)

5 kỳ vọng 5
4 kỳ vọng 4
0 kỳ vọng 0


**Độ phức tạp:** tới `max = 10⁵` giá trị `x`, mỗi giá trị `O(n)` ⇒ `O(max · n) = 10¹⁰` worst (khi đáp án nhỏ hoặc `0`). Space `O(1)`.

**Nút thắt:** ta thử **từng** `x` dù `count` đơn điệu — có thể nhảy cóc bằng nhị phân.

In [7]:
random.seed(0)
for n, M in [(100, 2000), (300, 5000)]:
    rib = [random.randint(1, M) for _ in range(n)]
    t0 = time.perf_counter(); r = max_length_linear(rib, 10**9)          # k rất lớn -> quét hết
    print(f"n={n}, max={M}, k=1e9: {r}, {time.perf_counter() - t0:.3f}s")

n=100, max=2000, k=1e9: 0, 0.005s
n=300, max=5000, k=1e9: 0, 0.035s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight (đơn điệu):** nếu cắt được `k` đoạn dài `x` thì cũng cắt được `k` đoạn dài `x' < x` (mỗi đoạn dài `x` cắt bớt thành `x'`). Hình thức: `⌊r/x'⌋ ≥ ⌊r/x⌋`. ⇒ `ok(x) = [count(x) ≥ k]` đúng trên một tiền tố `[1, X*]`.

**Tìm kiếm nhị phân trên đáp án:** tìm `X*` = giá trị **lớn nhất** có `ok` đúng, trên miền `[0, hi]` với `ok(0)` quy ước đúng (đáp án `0` khi không có `x ≥ 1` nào thỏa).

**Quy tắc biên:**
- `hi = min(max(ribbons), sum(ribbons) // k)`; nếu `hi == 0` ⇒ trả `0`.
- `mid = (lo + hi + 1) // 2`; `ok(mid)` ⇒ `lo = mid`, ngược lại `hi = mid − 1`.

**Insight 2 (đếm theo bội số):** với bảng đếm `cnt[v]` và tiền tố `C[v] = #{r ≤ v}`, `count(x) = Σ_{q ≥ 1} #{r ≥ q·x} = Σ_{q ≥ 1} (n − C[q·x − 1])`. Số hạng `q` chạy tới `V/x` ⇒ `O(V/x)`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tìm kiếm nhị phân trên `[0, max]`

In [8]:
def max_length_binary(ribbons, k):
    lo, hi = 0, max(ribbons)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if sum(r // mid for r in ribbons) >= k:
            lo = mid
        else:
            hi = mid - 1
    return lo

for (rib, k), want in examples:
    print(max_length_binary(rib, k), "kỳ vọng", want)

5 kỳ vọng 5
4 kỳ vọng 4
0 kỳ vọng 0


`lo = 0` là "đáp án dự phòng": nếu không `mid ≥ 1` nào thỏa, `hi` giảm dần về `0` và trả `0`. Vì `mid ≥ 1` luôn (do làm tròn lên với `hi > lo ≥ 0`), không bao giờ chia cho 0.

**Độ phức tạp:** `log₂(10⁵) ≈ 17` vòng × `O(n)` ⇒ `O(n log max)`. Space `O(1)`.

### 7.2 Cách 3 — Thu hẹp cận trên bằng tổng chiều dài

`hi = min(max(ribbons), sum(ribbons) // k)`. Nếu `sum < k` thì `hi = 0` ⇒ trả `0` ngay không cần vòng lặp. Khi `k` lớn, `hi` nhỏ hơn nhiều so với `max` ⇒ ít vòng hơn.

In [9]:
def max_length_bounded(ribbons, k):
    hi = min(max(ribbons), sum(ribbons) // k)
    lo = 0
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if sum(r // mid for r in ribbons) >= k:
            lo = mid
        else:
            hi = mid - 1
    return lo

for (rib, k), want in examples:
    print(max_length_bounded(rib, k), "kỳ vọng", want)

5 kỳ vọng 5
4 kỳ vọng 4
0 kỳ vọng 0


Code nộp LeetCode của Cách 3: xem mục 9.

**Độ phức tạp:** `O(n log hi)` với `hi ≤ min(max, Σ/k)`.

### 7.3 Cách 4 — Bảng đếm + tổng điều hòa, quét `x` giảm dần

Từng đoạn:
1. `cnt[v]` = số dải dài `v`; `ge[v]` = số dải dài `≥ v` (hậu tố).
2. `count(x) = Σ_{q=1}^{⌊V/x⌋} ge[q·x]` (mỗi dải dài `r` đóng góp 1 cho mọi `q` với `q·x ≤ r`, tổng `⌊r/x⌋`).
3. Quét `x` từ `V` xuống 1, trả về `x` đầu tiên có `count(x) ≥ k`.

In [10]:
def max_length_harmonic(ribbons, k):
    V = max(ribbons)
    ge = [0] * (V + 2)
    for r in ribbons:
        ge[r] += 1
    for v in range(V - 1, 0, -1):
        ge[v] += ge[v + 1]               # ge[v] = số dải dài >= v
    for x in range(V, 0, -1):
        total = 0
        for m in range(x, V + 1, x):     # m = q * x
            total += ge[m]
            if total >= k:
                return x
    return 0

for (rib, k), want in examples:
    print(max_length_harmonic(rib, k), "kỳ vọng", want)

5 kỳ vọng 5
4 kỳ vọng 4
0 kỳ vọng 0


**Độ phức tạp:** worst `Σ_x V/x = O(V log V)` + `O(n + V)`, Space `O(V)`. Không phụ thuộc `n` trong vòng quét — tốt khi `n` lớn và `V` nhỏ. Dừng sớm khi `total ≥ k`.

Bonus: cách này tính được `count(x)` cho **mọi** `x` cùng lúc — hữu ích khi có nhiều truy vấn `k` khác nhau.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 2 `[7, 5, 9]`, `k = 4`: `hi = min(9, 21 // 4) = 5`.

In [11]:
def trace_bounded(ribbons, k):
    hi = min(max(ribbons), sum(ribbons) // k); lo = 0
    print(f"cận trên hi = min(max={max(ribbons)}, sum//k={sum(ribbons) // k}) = {hi}")
    while lo < hi:
        mid = (lo + hi + 1) // 2
        c = sum(r // mid for r in ribbons)
        ok = c >= k
        print(f"  [lo={lo}, hi={hi}] mid={mid}: count={c} ({' + '.join(str(r // mid) for r in ribbons)}) -> "
              f"{'đủ, lo = mid' if ok else 'thiếu, hi = mid - 1'}")
        if ok: lo = mid
        else: hi = mid - 1
    print("đáp án:", lo)
    return lo

trace_bounded([7, 5, 9], 4)
trace_bounded([5, 7, 9], 22)

cận trên hi = min(max=9, sum//k=5) = 5
  [lo=0, hi=5] mid=3: count=6 (2 + 1 + 3) -> đủ, lo = mid
  [lo=3, hi=5] mid=4: count=4 (1 + 1 + 2) -> đủ, lo = mid
  [lo=4, hi=5] mid=5: count=3 (1 + 1 + 1) -> thiếu, hi = mid - 1
đáp án: 4
cận trên hi = min(max=9, sum//k=0) = 0
đáp án: 0


0

In [12]:
trace_bounded([100000, 1, 1], 7)

cận trên hi = min(max=100000, sum//k=14286) = 14286
  [lo=0, hi=14286] mid=7143: count=13 (13 + 0 + 0) -> đủ, lo = mid
  [lo=7143, hi=14286] mid=10715: count=9 (9 + 0 + 0) -> đủ, lo = mid
  [lo=10715, hi=14286] mid=12501: count=7 (7 + 0 + 0) -> đủ, lo = mid
  [lo=12501, hi=14286] mid=13394: count=7 (7 + 0 + 0) -> đủ, lo = mid
  [lo=13394, hi=14286] mid=13840: count=7 (7 + 0 + 0) -> đủ, lo = mid
  [lo=13840, hi=14286] mid=14063: count=7 (7 + 0 + 0) -> đủ, lo = mid
  [lo=14063, hi=14286] mid=14175: count=7 (7 + 0 + 0) -> đủ, lo = mid
  [lo=14175, hi=14286] mid=14231: count=7 (7 + 0 + 0) -> đủ, lo = mid
  [lo=14231, hi=14286] mid=14259: count=7 (7 + 0 + 0) -> đủ, lo = mid
  [lo=14259, hi=14286] mid=14273: count=7 (7 + 0 + 0) -> đủ, lo = mid
  [lo=14273, hi=14286] mid=14280: count=7 (7 + 0 + 0) -> đủ, lo = mid
  [lo=14280, hi=14286] mid=14283: count=7 (7 + 0 + 0) -> đủ, lo = mid
  [lo=14283, hi=14286] mid=14285: count=7 (7 + 0 + 0) -> đủ, lo = mid
  [lo=14285, hi=14286] mid=14286: count=6 

14285

Bảng tóm tắt trace đầu:

| lo | hi | mid | count | kết luận |
|---|---|---|---|---|
| 0 | 5 | 3 | 2+1+3 = 6 | đủ ⇒ lo = 3 |
| 3 | 5 | 4 | 1+1+2 = 4 | đủ ⇒ lo = 4 |
| 4 | 5 | 5 | 1+1+1 = 3 | thiếu ⇒ hi = 4 |
| 4 | 4 | — | — | **4** |

In [13]:
def trace_harmonic(ribbons, k):
    V = max(ribbons); ge = [0] * (V + 2)
    for r in ribbons: ge[r] += 1
    for v in range(V - 1, 0, -1): ge[v] += ge[v + 1]
    print("ge[v] (số dải dài >= v):", {v: ge[v] for v in range(1, V + 1)})
    for x in range(V, 0, -1):
        terms = [ge[m] for m in range(x, V + 1, x)]
        print(f"x={x}: count = {' + '.join(map(str, terms))} = {sum(terms)}")
        if sum(terms) >= k:
            print("-> đáp án", x); return x
    return 0

trace_harmonic([9, 7, 5], 3)

ge[v] (số dải dài >= v): {1: 3, 2: 3, 3: 3, 4: 3, 5: 3, 6: 2, 7: 2, 8: 1, 9: 1}
x=9: count = 1 = 1
x=8: count = 1 = 1
x=7: count = 2 = 2
x=6: count = 2 = 2
x=5: count = 3 = 3
-> đáp án 5


5

## 9. Code nộp LeetCode

In [14]:
class Solution:
    def maxLength(self, ribbons: List[int], k: int) -> int:
        lo, hi = 0, min(max(ribbons), sum(ribbons) // k)
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if sum(r // mid for r in ribbons) >= k:
                lo = mid
            else:
                hi = mid - 1
        return lo

sol = Solution()
for (rib, k), want in examples:
    got = sol.maxLength(rib, k)
    print(got, "OK" if got == want else "SAI")

5 OK
4 OK
0 OK


## 10. Vì sao thuật toán đúng

**Tính khả thi ⇔ `count(x) ≥ k`.** Với `x` cố định, cách cắt tối ưu là cắt mỗi dải thành nhiều đoạn dài `x` nhất có thể: `⌊rᵢ/x⌋` đoạn. Không cách nào cho nhiều đoạn dài `x` hơn từ dải đó. ⇒ cắt được `k` đoạn ⇔ `Σ ⌊rᵢ/x⌋ ≥ k`.

**Đơn điệu.** `x' < x ⇒ ⌊r/x'⌋ ≥ ⌊r/x⌋` ⇒ `count(x') ≥ count(x)`. Vậy `ok` đúng trên `[1, X*]`, sai trên `(X*, ∞)`.

**Cận trên đúng.** Nếu `ok(x)` với `x ≥ 1`: (i) có ít nhất một đoạn dài `x` từ một dải nào đó ⇒ `x ≤ max`; (ii) `k` đoạn dài `x` có tổng `k·x ≤ Σ rᵢ` ⇒ `x ≤ Σ/k`. ⇒ `X* ≤ hi`.

**Nhị phân.** Invariant: `ok(lo)` đúng (quy ước `ok(0)`), và mọi `x > hi` có `ok(x)` sai. Mỗi vòng giữ invariant và thu hẹp `[lo, hi]` (vì `mid > lo` khi `lo < hi`). Kết thúc `lo = hi = X*` (hoặc `0`).

**Cách 4.** `Σ_{q ≥ 1} ge[q x] = Σ_q #{r : r ≥ q x} = Σ_r #{q ≥ 1 : q x ≤ r} = Σ_r ⌊r/x⌋`. ✓

## 11. Phân tích Time / Space

Gọi `n ≤ 10⁵`, `M = max(ribbons) ≤ 10⁵`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Quét `x` giảm dần | `O(M · n)` | `O(1)` | kiểm chứng |
| 2. Nhị phân `[0, M]` | `O(n log M)` | `O(1)` | chuẩn |
| 3. Nhị phân + cận `Σ/k` | `O(n log min(M, Σ/k))` | `O(1)` | **nộp bài** |
| 4. Đếm + tổng điều hòa | `O(n + M log M)` | `O(M)` | nhiều truy vấn `k` |

In [15]:
random.seed(1)
rib = [random.randint(1, 10**5) for _ in range(10**5)]
for k in [1, 10**5, 10**7, 10**9]:
    line = f"k={k:>10}:"
    res = []
    for name, fn in [("binary", max_length_binary), ("bounded", max_length_bounded), ("harmonic", max_length_harmonic),
                     ("submit", Solution().maxLength)]:
        t0 = time.perf_counter(); res.append(fn(rib, k)); line += f" {name}={time.perf_counter() - t0:.3f}s"
    assert len(set(res)) == 1
    print(line, "-> x =", res[0])

k=         1: binary=0.038s bounded=0.039s harmonic=0.010s submit=0.039s -> x = 100000


k=    100000: binary=0.039s bounded=0.036s harmonic=0.015s submit=0.035s -> x = 33310


k=  10000000: binary=0.038s bounded=0.021s harmonic=0.042s submit=0.021s -> x = 497


k=1000000000: binary=0.043s bounded=0.009s harmonic=0.061s submit=0.009s -> x = 4


## 12. Bộ test

In [16]:
fns = [max_length_binary, max_length_bounded, max_length_harmonic, lambda a, k: Solution().maxLength(a, k)]

for (rib, k), want in examples:
    assert max_length_linear(rib, k) == want
    for fn in fns:
        assert fn(rib, k) == want

edge = [
    (([1], 1), 1), (([1], 2), 0), (([100000], 1), 100000), (([100000], 100000), 1),
    (([100000], 100001), 0), (([5, 5, 5], 3), 5), (([5, 5, 5], 4), 2), (([1, 1, 1], 3), 1),
    (([3, 4], 1), 4), (([10**5] * 3, 10**9), 0),
]
for (rib, k), want in edge:
    assert max_length_linear(rib, k) == want, (rib, k)
    for fn in fns:
        assert fn(rib, k) == want, (fn, rib, k)

random.seed(2024)
for _ in range(1500):
    n = random.randint(1, 8)
    M = random.choice([5, 30, 200])
    rib = [random.randint(1, M) for _ in range(n)]
    k = random.choice([1, random.randint(1, 10), random.randint(1, 200), 10**9])
    want = max_length_linear(rib, k)
    for fn in fns:
        assert fn(rib, k) == want, (fn, rib, k)
print("ví dụ + edge + 1500 bộ ngẫu nhiên khớp quét tuyến tính")

ví dụ + edge + 1500 bộ ngẫu nhiên khớp quét tuyến tính


In [17]:
# kiểm tra định nghĩa "lớn nhất": ok(X*) đúng và ok(X*+1) sai
random.seed(9)
for _ in range(300):
    rib = [random.randint(1, 1000) for _ in range(random.randint(1, 50))]
    k = random.randint(1, 3000)
    x = Solution().maxLength(rib, k)
    if x > 0:
        assert count(rib, x) >= k
    assert count(rib, x + 1) < k
print("X* thỏa và X*+1 không thỏa (300 bộ)")

X* thỏa và X*+1 không thỏa (300 bộ)


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Nhị phân "tìm lớn nhất" nhưng làm tròn xuống ⇒ vòng lặp vô hạn

Minh họa an toàn bằng giới hạn số vòng:

In [18]:
def wrong_round_down(ribbons, k, max_iter=100):
    lo, hi = 0, max(ribbons); it = 0
    while lo < hi and it < max_iter:
        mid = (lo + hi) // 2                       # SAI: làm tròn xuống
        if mid == 0 or sum(r // mid for r in ribbons) >= k:
            lo = mid
        else:
            hi = mid - 1
        it += 1
    return lo, it

res, it = wrong_round_down([9, 7, 5], 3)
print(f"dừng sau {it} vòng (giới hạn 100) với lo = {res} -> {'KẸT' if it == 100 else 'ok'}")

dừng sau 100 vòng (giới hạn 100) với lo = 4 -> KẸT


Khi `hi = lo + 1` và `ok(lo)` đúng, `mid = lo` ⇒ `lo = mid` không đổi ⇒ lặp mãi. **Sửa:** `mid = (lo + hi + 1) // 2`.

### Lỗi 2 — Bắt đầu nhị phân từ `lo = 1` và quên trường hợp không thể

In [19]:
def wrong_lo_one(ribbons, k):
    lo, hi = 1, max(ribbons)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if sum(r // mid for r in ribbons) >= k:
            lo = mid
        else:
            hi = mid - 1
    return lo                                      # SAI: trả 1 dù count(1) < k

print("sai =", wrong_lo_one([5, 7, 9], 22), "| đúng =", Solution().maxLength([5, 7, 9], 22))

sai = 1 | đúng = 0


**Sửa:** `lo = 0` (đáp án dự phòng), hoặc kiểm tra `count(1) < k ⇒ 0` trước.

### Lỗi 3 — Dùng tổng chiều dài chia `k` làm đáp án

In [20]:
def wrong_average(ribbons, k):
    return sum(ribbons) // k                       # SAI: không tính phần dư của từng dải

for (rib, k), want in examples:
    print("sai =", wrong_average(rib, k), "| đúng =", want)

sai = 7 | đúng = 5
sai = 5 | đúng = 4
sai = 0 | đúng = 0


`Σ/k` chỉ là **cận trên**: phần dư `r mod x` của từng dải bị bỏ, không ghép được. **Sửa:** dùng làm `hi` cho nhị phân.

### Lỗi 4 — Ghép các phần dư của nhiều dải (hiểu sai đề)

Ví dụ coi `[3, 3]` với `x = 2` cho `3` đoạn (vì `6 / 2 = 3`) — nhưng phần dư `1 + 1` không được nối lại.

In [21]:
def wrong_join_leftovers(ribbons, k):
    for x in range(max(ribbons), 0, -1):
        if sum(ribbons) // x >= k:                 # SAI: coi như nối được phần dư
            return x
    return 0

for rib, k in [([3, 3], 3), ([9, 7, 5], 3), ([1, 1, 5], 2)]:
    print(f"{rib}, k={k}: sai = {wrong_join_leftovers(rib, k)} | đúng = {Solution().maxLength(rib, k)}")

[3, 3], k=3: sai = 2 | đúng = 1
[9, 7, 5], k=3: sai = 7 | đúng = 5
[1, 1, 5], k=2: sai = 3 | đúng = 2


**Sửa:** đếm từng dải riêng `Σ ⌊rᵢ/x⌋`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tìm giá trị lớn nhất/nhỏ nhất sao cho **có thể** …" + hàm kiểm tra đơn điệu ⇒ **nhị phân trên đáp án**.
- "Lớn nhất thỏa" ⇒ `mid` làm tròn **lên**; "nhỏ nhất thỏa" ⇒ làm tròn **xuống**.
- Luôn có "đáp án dự phòng" (`0` hoặc `−1`) ở một đầu khoảng.
- Cận chặt (`Σ/k`) giúp trả lời nhanh các trường hợp không thể.
- Họ bài: 875 *Koko Eating Bananas*, 1011 *Capacity To Ship Packages*, 410 *Split Array Largest Sum*, 1482 *Minimum Days to Make m Bouquets*, 2226 *Maximum Candies Allocated to K Children* (gần như y hệt).

## 15. Biến thể của bài

1. **2226 Maximum Candies Allocated to K Children:** y hệt (đống kẹo = dải ruy-băng).
2. **Độ dài thực (cắt được độ dài không nguyên):** nhị phân trên số thực với sai số `ε`, hoặc lặp cố định số vòng.
3. **Mỗi dải chỉ được cắt tối đa `c` đoạn:** `count(x) = Σ min(⌊r/x⌋, c)` — vẫn đơn điệu.
4. **Nhiều truy vấn `k`:** tiền xử lý `count(x)` cho mọi `x` bằng Cách 4, rồi nhị phân trên mảng `count` (không tăng) cho mỗi `k`.

In [22]:
def max_length_real(ribbons, k, iters=60):
    """Biến thể 2: độ dài thực lớn nhất (cắt được độ dài bất kỳ > 0)."""
    lo, hi = 0.0, float(max(ribbons))
    for _ in range(iters):
        mid = (lo + hi) / 2
        if sum(int(r // mid) for r in ribbons) >= k:
            lo = mid
        else:
            hi = mid
    return lo

def all_counts(ribbons):
    """Biến thể 4: count(x) cho mọi x bằng tổng điều hòa."""
    V = max(ribbons); ge = [0] * (V + 2)
    for r in ribbons: ge[r] += 1
    for v in range(V - 1, 0, -1): ge[v] += ge[v + 1]
    return [0] + [sum(ge[m] for m in range(x, V + 1, x)) for x in range(1, V + 1)]

rib = [9, 7, 5]
print("độ dài thực, k=4:", round(max_length_real(rib, 4), 6), "(nguyên: 4)")
cnts = all_counts(rib)
print("count(x):", cnts[1:])
from bisect import bisect_left
# count không tăng; đáp án cho k = số x lớn nhất có count(x) >= k
for k in [1, 3, 4, 6, 21, 22]:
    x = max((x for x in range(1, len(cnts)) if cnts[x] >= k), default=0)
    assert x == Solution().maxLength(rib, k)
    print(f"k={k}: x={x}")

độ dài thực, k=4: 4.5 (nguyên: 4)
count(x): [21, 9, 6, 4, 3, 2, 2, 1, 1]
k=1: x=9
k=3: x=5
k=4: x=4
k=6: x=3
k=21: x=1
k=22: x=0


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét giảm dần | thử từng `x` | `O(M n)` | `O(1)` | kiểm chứng |
| 2. Nhị phân `[0, M]` | đơn điệu | `O(n log M)` | `O(1)` | chuẩn |
| 3. Nhị phân + cận | `hi = min(M, Σ/k)` | `O(n log hi)` | `O(1)` | **nộp bài** |
| 4. Tổng điều hòa | đếm theo bội số | `O(n + M log M)` | `O(M)` | nhiều truy vấn |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `maxLength([1, 2, 3, 4, 9], 5)` và `maxLength([10], 3)`?

**Bài 2 (tính tay):** chạy nhị phân bằng tay cho `ribbons = [12, 8, 6]`, `k = 5` (ghi `lo, hi, mid, count`).

**Bài 3 (điền TODO):** hoàn thiện `max_length_capped(ribbons, k, c)` — biến thể mỗi dải cắt tối đa `c` đoạn.

**Bài 4 (tự viết lại):** viết lại Cách 3 dưới dạng khuôn "tìm `x` **nhỏ nhất** không thỏa" (`first_false`) rồi trả `first_false − 1`.

**Bài 5 (mini challenge):** cho nhiều truy vấn `k₁, …, k_q` trên cùng `ribbons`. Viết `answer_queries(ribbons, ks)` trong `O(n + M log M + q log M)` và kiểm chứng với lời giải từng truy vấn.

In [23]:
def max_length_capped(ribbons, k, c):
    # TODO: count(x) = sum(min(r // x, c)); nhị phân lớn nhất
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `x = 3`: `0+0+1+1+3 = 5` ⇒ đủ. (2) `x = 4`: `0+0+0+1+2 = 3` ⇒ thiếu. (3) Đáp án `3`; `[10]`, `k = 3`: `x = 3` cho `3` đoạn ⇒ `3`.

**Bài 2:** (1) `hi = min(12, 26 // 5) = 5`. (2) `mid = 3`: `4+2+2 = 8` đủ ⇒ `lo = 3`; `mid = 4`: `3+2+1 = 6` đủ ⇒ `lo = 4`. (3) `mid = 5`: `2+1+1 = 4` thiếu ⇒ `hi = 4` ⇒ `4`.

**Bài 3:** (1) Hàm đếm mới vẫn không tăng theo `x`. (2) Cận trên `max`. (3) Khuôn nhị phân giữ nguyên.

**Bài 4:** (1) `first_false` trên `[1, hi + 1]`. (2) `mid = (lo + hi) // 2`. (3) Trả `first_false − 1`.

**Bài 5:** (1) `cnts = all_counts(ribbons)` (không tăng). (2) Với mỗi `k`, tìm `x` lớn nhất có `cnts[x] ≥ k` bằng nhị phân trên chỉ số. (3) Nếu `cnts[1] < k` ⇒ `0`.

## 19. Đáp án tham khảo

In [24]:
# Bài 1
assert Solution().maxLength([1, 2, 3, 4, 9], 5) == 3 and Solution().maxLength([10], 3) == 3

# Bài 2
trace_bounded([12, 8, 6], 5)
assert Solution().maxLength([12, 8, 6], 5) == 4

# Bài 3
def max_length_capped_ans(ribbons, k, c):
    lo, hi = 0, max(ribbons)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if sum(min(r // mid, c) for r in ribbons) >= k:
            lo = mid
        else:
            hi = mid - 1
    return lo

# Bài 4
def max_length_first_false(ribbons, k):
    lo, hi = 1, min(max(ribbons), sum(ribbons) // k) + 1       # tìm x nhỏ nhất mà count(x) < k
    while lo < hi:
        mid = (lo + hi) // 2
        if sum(r // mid for r in ribbons) < k:
            hi = mid
        else:
            lo = mid + 1
    return lo - 1

random.seed(211)
for _ in range(800):
    rib = [random.randint(1, 60) for _ in range(random.randint(1, 8))]; k = random.randint(1, 80); c = random.randint(1, 5)
    brute_c = max((x for x in range(1, max(rib) + 1) if sum(min(r // x, c) for r in rib) >= k), default=0)
    assert max_length_capped_ans(rib, k, c) == brute_c
    assert max_length_first_false(rib, k) == max_length_linear(rib, k)
print("Bài 1-4: OK")

cận trên hi = min(max=12, sum//k=5) = 5
  [lo=0, hi=5] mid=3: count=8 (4 + 2 + 2) -> đủ, lo = mid
  [lo=3, hi=5] mid=4: count=6 (3 + 2 + 1) -> đủ, lo = mid
  [lo=4, hi=5] mid=5: count=4 (2 + 1 + 1) -> thiếu, hi = mid - 1
đáp án: 4
Bài 1-4: OK


In [25]:
# Bài 5 — nhiều truy vấn
def answer_queries(ribbons, ks):
    cnts = all_counts(ribbons)            # cnts[x], x = 1..M, không tăng
    M = len(cnts) - 1
    out = []
    for k in ks:
        lo, hi = 0, M
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if cnts[mid] >= k: lo = mid
            else: hi = mid - 1
        out.append(lo)
    return out

random.seed(212)
for _ in range(100):
    rib = [random.randint(1, 300) for _ in range(random.randint(1, 30))]
    ks = [random.randint(1, 500) for _ in range(20)]
    assert answer_queries(rib, ks) == [Solution().maxLength(rib, k) for k in ks]
print("Bài 5: OK;", answer_queries([9, 7, 5], [1, 2, 3, 4, 5, 21, 22]))

Bài 5: OK; [9, 7, 5, 4, 3, 1, 0]


## 20. Template nhớ nhanh + Checklist

```python
# Nhị phân trên đáp án: x LỚN NHẤT với count(x) = sum(r // x) >= k
lo, hi = 0, min(max(ribbons), sum(ribbons) // k)   # lo = 0 là đáp án dự phòng
while lo < hi:
    mid = (lo + hi + 1) // 2                       # làm tròn LÊN
    if sum(r // mid for r in ribbons) >= k:
        lo = mid
    else:
        hi = mid - 1
return lo
```

**Checklist trước khi submit:**
- [ ] Hàm đếm theo **từng dải** (không nối phần dư).
- [ ] `mid` làm tròn lên khi tìm giá trị lớn nhất.
- [ ] `lo = 0` để trả `0` khi không thể (và không bao giờ chia cho 0).
- [ ] Cận trên `min(max, Σ/k)`.
- [ ] Thử `k` rất lớn (`10⁹`) và `k = 1` (đáp án `max`).